# Error Handling & Resilience — Agents That Don't Break

## What This Notebook Teaches You

Agents fail. LLMs produce malformed output, tools crash, APIs timeout, and multi-step reasoning goes off the rails. A production agent must **recover gracefully** from all of these.

This notebook builds five resilience patterns:

1. **RetryWithFeedback** — on error, re-prompt with the error message so the LLM can self-correct
2. **FallbackChain** — try strategy A, if it fails try B, then C
3. **CircuitBreaker** — after N consecutive failures, stop trying that path
4. **GracefulDegradation** — return partial results when the full task fails
5. **Timeout handling** — wrap tool calls with timeout protection

Then we combine all patterns into a **resilient agent** and test it against deliberately failing scenarios.

---

> **Prerequisites**: Modules 1–5 (ReAct agent patterns).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~45–60 minutes.

## Part 0: Environment Setup

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. Why Resilience Matters

### How Agents Fail

| Failure Mode | Example | Frequency |
|-------------|---------|-----------|
| **Malformed output** | LLM doesn't follow the Action/Action Input format | Common |
| **Tool error** | Calculator gets invalid expression, API returns error | Common |
| **Timeout** | Tool call takes too long, network hangs | Moderate |
| **Infinite loop** | Agent repeats the same action endlessly | Moderate |
| **Hallucinated tool** | Agent invokes a tool that doesn't exist | Occasional |
| **Context overflow** | Conversation too long for model's context window | Rare |

Without resilience, **any single failure crashes the entire agent**. With resilience, the agent recovers and still delivers useful results.

### The Resilience Stack

```
User Query
  → Timeout Protection (prevent hangs)
    → CircuitBreaker (stop repeated failures)
      → FallbackChain (try alternative strategies)
        → RetryWithFeedback (self-correct on errors)
          → GracefulDegradation (return partial results)
            → Agent Logic + Tools
```

## 2. RetryWithFeedback — Self-Correcting Agents

The simplest resilience pattern: when something fails, **tell the LLM what went wrong** and ask it to try again. LLMs are surprisingly good at self-correction when given error feedback.

In [ ]:
class RetryWithFeedback:
    """Retry failed operations by feeding error messages back to the LLM."""

    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries
        self.retry_log: List[Dict] = []

    def execute(self, fn: Callable, messages: List[Dict],
                error_prefix: str = "Your previous attempt failed") -> Tuple[Any, int]:
        """Execute a function with retry-on-error and feedback.

        Args:
            fn: Function that takes messages and returns (result, success).
            messages: Current conversation messages.
            error_prefix: Prefix for error feedback message.

        Returns:
            (result, num_retries) tuple.
        """
        last_error = None

        for attempt in range(self.max_retries + 1):
            try:
                result, success = fn(messages)
                if success:
                    self.retry_log.append({
                        "attempt": attempt + 1,
                        "status": "success",
                    })
                    return result, attempt
                else:
                    last_error = result  # result contains error info on failure
                    # Add error feedback to messages
                    feedback = (
                        f"{error_prefix} with error: {last_error}\n"
                        f"Please try again with a corrected approach."
                    )
                    messages = messages + [
                        {"role": "assistant", "content": str(last_error)},
                        {"role": "user", "content": feedback},
                    ]
            except Exception as e:
                last_error = str(e)
                feedback = (
                    f"{error_prefix} with exception: {last_error}\n"
                    f"Please try again, avoiding the error."
                )
                messages = messages + [
                    {"role": "user", "content": feedback},
                ]

            self.retry_log.append({
                "attempt": attempt + 1,
                "status": "retry",
                "error": str(last_error)[:100],
            })

        return last_error, self.max_retries + 1

    def stats(self) -> str:
        successes = sum(1 for r in self.retry_log if r["status"] == "success")
        retries = sum(1 for r in self.retry_log if r["status"] == "retry")
        return f"RetryWithFeedback: {successes} successes, {retries} retries across {len(self.retry_log)} operations"

# Demo: Show retry with feedback on a malformed tool call
retry = RetryWithFeedback(max_retries=2)

def simulate_parse_action(messages):
    """Simulate parsing an agent response that might be malformed."""
    response = generate(messages, max_new_tokens=128, temperature=0.3, do_sample=True)

    # Try to parse Action format
    action_match = re.search(r'Action:\s*(\w+)', response)
    input_match = re.search(r'Action Input:\s*(.+)', response)

    if action_match and input_match:
        return {"action": action_match.group(1), "input": input_match.group(1),
                "raw": response}, True
    else:
        return f"Could not parse action from response: '{response[:100]}'", False

messages = [
    {"role": "system", "content": "You have tools: calculator (math expressions), knowledge_base (fact lookup).\nAlways respond with:\nThought: [reasoning]\nAction: [tool_name]\nAction Input: [input]"},
    {"role": "user", "content": "What is 347 * 23?"},
]

result, retries = retry.execute(simulate_parse_action, messages)
print(f"Result after {retries} retries:")
if isinstance(result, dict):
    print(f"  Action: {result['action']}")
    print(f"  Input: {result['input']}")
else:
    print(f"  Failed: {result}")
print(f"\n{retry.stats()}")

## 3. FallbackChain — Multiple Strategies

When one approach fails, try another. This is especially useful when you have multiple ways to solve a problem — different prompts, different tools, or different decompositions.

In [ ]:
class FallbackChain:
    """Try multiple strategies in order until one succeeds."""

    def __init__(self, strategies: List[Dict]):
        """
        Args:
            strategies: List of dicts with keys:
                'name': str, 'fn': Callable that returns (result, success)
        """
        self.strategies = strategies
        self.execution_log: List[Dict] = []

    def execute(self, *args, **kwargs) -> Tuple[Any, str]:
        """Try each strategy in order. Return (result, strategy_name).

        Returns (None, "all_failed") if every strategy fails.
        """
        for strategy in self.strategies:
            name = strategy["name"]
            fn = strategy["fn"]
            try:
                result, success = fn(*args, **kwargs)
                self.execution_log.append({
                    "strategy": name,
                    "status": "success" if success else "failed",
                })
                if success:
                    return result, name
            except Exception as e:
                self.execution_log.append({
                    "strategy": name,
                    "status": "error",
                    "error": str(e)[:100],
                })
                continue

        return None, "all_failed"

    def stats(self) -> str:
        by_strategy = defaultdict(lambda: {"success": 0, "failed": 0, "error": 0})
        for entry in self.execution_log:
            by_strategy[entry["strategy"]][entry["status"]] += 1
        parts = [f"{name}: {dict(counts)}" for name, counts in by_strategy.items()]
        return "FallbackChain: " + " | ".join(parts)

# Demo: Three strategies for answering a math question
def strategy_direct(question: str):
    """Strategy A: Ask the LLM directly."""
    messages = [{"role": "user", "content": f"Answer concisely: {question}"}]
    response = generate(messages, max_new_tokens=64, temperature=0.1, do_sample=True)
    # Check if it looks like a number
    numbers = re.findall(r'[\d.]+', response)
    if numbers:
        return {"answer": numbers[0], "strategy": "direct"}, True
    return "No numeric answer found", False

def strategy_calculator(question: str):
    """Strategy B: Extract expression and use calculator."""
    messages = [{"role": "user", "content":
        f"Extract ONLY the math expression from this question (no words, just the expression): {question}"}]
    response = generate(messages, max_new_tokens=32, temperature=0.1, do_sample=True)
    expr = response.strip()
    try:
        allowed = {'abs': abs, 'round': round, 'sqrt': math.sqrt, 'pow': pow}
        result = eval(expr, {"__builtins__": {}}, allowed)
        return {"answer": str(result), "strategy": "calculator"}, True
    except:
        return f"Could not evaluate: {expr}", False

def strategy_step_by_step(question: str):
    """Strategy C: Chain-of-thought step by step."""
    messages = [{"role": "user", "content":
        f"Solve step by step, then give ONLY the final number on the last line:\n{question}"}]
    response = generate(messages, max_new_tokens=256, temperature=0.3, do_sample=True)
    lines = response.strip().split("\n")
    last_line = lines[-1].strip()
    numbers = re.findall(r'[\d.]+', last_line)
    if numbers:
        return {"answer": numbers[-1], "strategy": "step_by_step"}, True
    return f"No final number found in: {last_line}", False

chain = FallbackChain([
    {"name": "direct", "fn": strategy_direct},
    {"name": "calculator", "fn": strategy_calculator},
    {"name": "step_by_step", "fn": strategy_step_by_step},
])

questions = [
    "What is 347 * 23?",
    "What is the square root of 1764?",
    "If a rectangle has area 120 and width 8, what is its length?",
]

print("FallbackChain Demo:")
print("=" * 60)
for q in questions:
    result, strategy_name = chain.execute(q)
    if result:
        print(f"  Q: {q}")
        print(f"  A: {result['answer']} (via strategy: {strategy_name})")
    else:
        print(f"  Q: {q} → ALL STRATEGIES FAILED")
    print()

print(chain.stats())

## 4. CircuitBreaker — Fail Fast on Repeated Errors

The circuit breaker pattern prevents wasting resources on a path that keeps failing. After N consecutive failures, the circuit "opens" and immediately rejects further attempts — until a cooldown period passes.

States:
- **CLOSED** (normal): Requests pass through. Failures increment counter.
- **OPEN** (tripped): Requests immediately fail. After cooldown, move to HALF_OPEN.
- **HALF_OPEN** (probing): Allow one request through. If it succeeds → CLOSED. If it fails → OPEN.

In [ ]:
class CircuitBreaker:
    """Circuit breaker to prevent repeated failing operations."""

    CLOSED = "closed"       # Normal operation
    OPEN = "open"           # Failing — reject all calls
    HALF_OPEN = "half_open" # Testing — allow one probe

    def __init__(self, name: str, failure_threshold: int = 3,
                 cooldown_seconds: float = 30.0):
        self.name = name
        self.failure_threshold = failure_threshold
        self.cooldown_seconds = cooldown_seconds
        self.state = self.CLOSED
        self.failure_count = 0
        self.last_failure_time = 0.0
        self.total_calls = 0
        self.total_blocked = 0
        self.state_history: List[Dict] = []

    def _transition(self, new_state: str, reason: str):
        self.state_history.append({
            "from": self.state, "to": new_state,
            "reason": reason, "time": time.time(),
        })
        self.state = new_state

    def can_execute(self) -> bool:
        """Check if the circuit allows execution."""
        self.total_calls += 1

        if self.state == self.CLOSED:
            return True
        elif self.state == self.OPEN:
            # Check cooldown
            elapsed = time.time() - self.last_failure_time
            if elapsed >= self.cooldown_seconds:
                self._transition(self.HALF_OPEN, f"Cooldown elapsed ({elapsed:.0f}s)")
                return True  # Allow probe
            self.total_blocked += 1
            return False
        elif self.state == self.HALF_OPEN:
            return True  # Allow the probe request
        return False

    def record_success(self):
        """Record a successful execution."""
        if self.state == self.HALF_OPEN:
            self._transition(self.CLOSED, "Probe succeeded")
        self.failure_count = 0

    def record_failure(self):
        """Record a failed execution."""
        self.failure_count += 1
        self.last_failure_time = time.time()

        if self.state == self.HALF_OPEN:
            self._transition(self.OPEN, "Probe failed")
        elif self.failure_count >= self.failure_threshold:
            self._transition(self.OPEN, f"{self.failure_count} consecutive failures")

    def execute(self, fn: Callable, *args, **kwargs) -> Tuple[Any, bool]:
        """Execute function through the circuit breaker.

        Returns (result, success). Returns (None, False) if circuit is open.
        """
        if not self.can_execute():
            return None, False

        try:
            result = fn(*args, **kwargs)
            self.record_success()
            return result, True
        except Exception as e:
            self.record_failure()
            return str(e), False

    def stats(self) -> str:
        return (
            f"CircuitBreaker '{self.name}': state={self.state} | "
            f"failures={self.failure_count}/{self.failure_threshold} | "
            f"calls={self.total_calls} | blocked={self.total_blocked} | "
            f"transitions={len(self.state_history)}"
        )

# Demo: Circuit breaker in action
breaker = CircuitBreaker("calculator", failure_threshold=3, cooldown_seconds=2.0)

def flaky_calculator(expr: str) -> str:
    """Simulated flaky tool that fails intermittently."""
    import random
    if random.random() < 0.7:  # 70% failure rate
        raise ValueError(f"Connection timeout for: {expr}")
    allowed = {'abs': abs, 'round': round, 'sqrt': math.sqrt}
    return str(eval(expr, {"__builtins__": {}}, allowed))

print("CircuitBreaker Demo (flaky calculator with 70% failure rate):")
print("-" * 60)

import random
random.seed(42)  # reproducible demo

for i in range(8):
    result, success = breaker.execute(flaky_calculator, "2 + 2")
    status = "✓" if success else ("⊘ BLOCKED" if breaker.state == "open" else "✗ FAILED")
    print(f"  Call {i+1}: {status} (state={breaker.state}, result={result})")
    if i == 5:
        print(f"  --- Waiting for cooldown ({breaker.cooldown_seconds}s) ---")
        time.sleep(breaker.cooldown_seconds + 0.1)

print(f"\n{breaker.stats()}")
print(f"State transitions:")
for t in breaker.state_history:
    print(f"  {t['from']} → {t['to']}: {t['reason']}")

## 5. GracefulDegradation — Partial Results Are Better Than Nothing

When a multi-step task partially fails, it's better to return what we have than to return nothing. GracefulDegradation captures partial results and returns them with a degradation notice.

In [ ]:
class GracefulDegradation:
    """Return partial results when full task completion fails."""

    def __init__(self):
        self.partial_results: List[Dict] = []
        self.failed_steps: List[Dict] = []

    def add_result(self, step_name: str, result: Any):
        """Record a successful partial result."""
        self.partial_results.append({
            "step": step_name,
            "result": result,
            "status": "success",
        })

    def add_failure(self, step_name: str, error: str):
        """Record a failed step."""
        self.failed_steps.append({
            "step": step_name,
            "error": error,
            "status": "failed",
        })

    @property
    def completion_rate(self) -> float:
        total = len(self.partial_results) + len(self.failed_steps)
        return len(self.partial_results) / total if total > 0 else 0.0

    def get_result(self) -> Dict:
        """Get the best available result."""
        if not self.failed_steps:
            return {
                "status": "complete",
                "results": [r["result"] for r in self.partial_results],
                "completion_rate": 1.0,
            }
        elif self.partial_results:
            return {
                "status": "partial",
                "results": [r["result"] for r in self.partial_results],
                "failed_steps": [f["step"] for f in self.failed_steps],
                "completion_rate": self.completion_rate,
                "message": f"Completed {len(self.partial_results)}/{len(self.partial_results)+len(self.failed_steps)} steps.",
            }
        else:
            return {
                "status": "failed",
                "results": [],
                "failed_steps": [f["step"] for f in self.failed_steps],
                "completion_rate": 0.0,
                "message": "All steps failed.",
            }

# Demo: Multi-step task with partial failures
degradation = GracefulDegradation()

# Simulate a multi-step research task
steps = [
    ("fetch_data", lambda: "GDP data for 2023: $25.5 trillion"),
    ("analyze_trend", lambda: "GDP grew 2.5% year-over-year"),
    ("fetch_comparison", lambda: (_ for _ in ()).throw(ConnectionError("API timeout"))),
    ("generate_summary", lambda: "The US economy showed moderate growth in 2023."),
]

print("GracefulDegradation Demo:")
print("-" * 50)
for step_name, fn in steps:
    try:
        result = fn()
        degradation.add_result(step_name, result)
        print(f"  ✓ {step_name}: {str(result)[:60]}")
    except Exception as e:
        degradation.add_failure(step_name, str(e))
        print(f"  ✗ {step_name}: {e}")

final = degradation.get_result()
print(f"\nFinal result:")
print(f"  Status: {final['status']}")
print(f"  Completion: {final['completion_rate']:.0%}")
if final.get("results"):
    print(f"  Results ({len(final['results'])}):")
    for r in final["results"]:
        print(f"    - {str(r)[:70]}")
if final.get("failed_steps"):
    print(f"  Failed steps: {final['failed_steps']}")
if final.get("message"):
    print(f"  Message: {final['message']}")

## 6. Timeout Handling

Tool calls can hang indefinitely — especially when calling external APIs. We build a timeout decorator that wraps any function with a time limit.

> **Note**: In Colab (Linux), we use `signal.alarm` for timeout. In other environments, threading-based timeout is an alternative.

In [ ]:
import signal
import functools

class TimeoutError(Exception):
    """Raised when a function call exceeds its time limit."""
    pass

def with_timeout(seconds: float):
    """Decorator that adds a timeout to any function.

    Uses signal-based timeout (Unix/Linux only — works in Colab).
    """
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            def handler(signum, frame):
                raise TimeoutError(
                    f"Function '{fn.__name__}' timed out after {seconds}s"
                )

            old_handler = signal.signal(signal.SIGALRM, handler)
            signal.alarm(int(seconds))
            try:
                result = fn(*args, **kwargs)
                signal.alarm(0)  # Cancel alarm
                return result
            except TimeoutError:
                raise
            finally:
                signal.signal(signal.SIGALRM, old_handler)

        return wrapper
    return decorator

# Demo: Timeout on a slow function
@with_timeout(2)
def slow_tool(query: str) -> str:
    """Simulated slow tool that takes a long time."""
    time.sleep(5)  # This will be interrupted
    return f"Result for {query}"

@with_timeout(2)
def fast_tool(query: str) -> str:
    """Simulated fast tool."""
    time.sleep(0.1)
    return f"Quick result for {query}"

print("Timeout Handling Demo:")
print("-" * 50)

# Fast tool — should succeed
try:
    result = fast_tool("test query")
    print(f"  ✓ fast_tool: {result}")
except TimeoutError as e:
    print(f"  ✗ fast_tool: {e}")

# Slow tool — should timeout
try:
    result = slow_tool("test query")
    print(f"  ✓ slow_tool: {result}")
except TimeoutError as e:
    print(f"  ✗ slow_tool: {e}")
    print(f"    → Agent can now try a fallback or return partial results")

## 7. Safe Tool Execution Wrapper

Combine timeout + retry + circuit breaker into a single tool execution wrapper.

In [ ]:
class SafeToolExecutor:
    """Execute tools with timeout, retry, and circuit breaker protection."""

    def __init__(self, timeout_seconds: float = 10.0, max_retries: int = 2,
                 circuit_threshold: int = 3):
        self.timeout_seconds = timeout_seconds
        self.max_retries = max_retries
        self.breakers: Dict[str, CircuitBreaker] = {}
        self.circuit_threshold = circuit_threshold
        self.execution_log: List[Dict] = []

    def _get_breaker(self, tool_name: str) -> CircuitBreaker:
        if tool_name not in self.breakers:
            self.breakers[tool_name] = CircuitBreaker(
                tool_name, failure_threshold=self.circuit_threshold
            )
        return self.breakers[tool_name]

    def execute(self, tool_name: str, tool_fn: Callable,
                *args, **kwargs) -> Tuple[Any, bool]:
        """Execute a tool with all safety wrappers.

        Returns (result, success).
        """
        breaker = self._get_breaker(tool_name)

        # Check circuit breaker
        if not breaker.can_execute():
            self.execution_log.append({
                "tool": tool_name, "status": "circuit_open",
            })
            return f"Circuit open for {tool_name}", False

        # Retry loop
        for attempt in range(self.max_retries + 1):
            try:
                # Apply timeout via signal (Colab/Linux)
                def handler(signum, frame):
                    raise TimeoutError(f"{tool_name} timed out")

                old = signal.signal(signal.SIGALRM, handler)
                signal.alarm(int(self.timeout_seconds))
                try:
                    result = tool_fn(*args, **kwargs)
                    signal.alarm(0)
                except TimeoutError:
                    raise
                finally:
                    signal.signal(signal.SIGALRM, old)

                breaker.record_success()
                self.execution_log.append({
                    "tool": tool_name, "status": "success", "attempts": attempt + 1,
                })
                return result, True

            except Exception as e:
                if attempt == self.max_retries:
                    breaker.record_failure()
                    self.execution_log.append({
                        "tool": tool_name, "status": "failed",
                        "error": str(e)[:100], "attempts": attempt + 1,
                    })
                    return str(e), False

        return "Max retries exceeded", False

    def stats(self) -> str:
        success = sum(1 for e in self.execution_log if e["status"] == "success")
        failed = sum(1 for e in self.execution_log if e["status"] == "failed")
        blocked = sum(1 for e in self.execution_log if e["status"] == "circuit_open")
        return f"SafeToolExecutor: {success} success, {failed} failed, {blocked} blocked"

# Demo
executor = SafeToolExecutor(timeout_seconds=3, max_retries=1, circuit_threshold=2)

def good_calculator(expr: str) -> str:
    allowed = {'abs': abs, 'round': round, 'sqrt': math.sqrt}
    return str(eval(expr, {"__builtins__": {}}, allowed))

def bad_calculator(expr: str) -> str:
    raise ValueError("Simulated tool crash")

print("SafeToolExecutor Demo:")
print("-" * 50)

# Good tool call
result, ok = executor.execute("calculator", good_calculator, "2 + 2")
print(f"  Good call: result={result}, success={ok}")

# Bad tool calls (will trip circuit breaker)
for i in range(3):
    result, ok = executor.execute("bad_calc", bad_calculator, "2 + 2")
    status = "blocked" if "Circuit open" in str(result) else ("ok" if ok else "failed")
    print(f"  Bad call {i+1}: result={str(result)[:50]}, status={status}")

print(f"\n{executor.stats()}")

## 8. Resilient Agent — All Patterns Combined

Now we wrap a ReAct agent with all resilience patterns and test it against deliberately failing scenarios.

In [ ]:
# Define tools including some that can fail
def calculator(expression: str) -> str:
    allowed = {'abs': abs, 'round': round, 'min': min, 'max': max,
               'pow': pow, 'sqrt': math.sqrt, 'pi': math.pi}
    return str(eval(expression, {"__builtins__": {}}, allowed))

def knowledge_base(query: str) -> str:
    kb = {
        "capital of france": "Paris",
        "chemical symbol gold": "Au",
        "who wrote 1984": "George Orwell",
        "boiling point water": "100°C (212°F)",
        "planets solar system": "8 planets",
    }
    query_lower = query.lower()
    for key, value in kb.items():
        if any(w in query_lower for w in key.split()):
            return value
    return f"Not found: {query}"

TOOLS = {
    "calculator": {"fn": calculator, "description": "Evaluate math expressions."},
    "knowledge_base": {"fn": knowledge_base, "description": "Look up facts."},
}

class ResilientAgent:
    """ReAct agent with full resilience stack."""

    def __init__(self, tools: Dict, max_steps: int = 5, max_retries: int = 2):
        self.tools = tools
        self.max_steps = max_steps
        self.retry = RetryWithFeedback(max_retries=max_retries)
        self.fallback = FallbackChain([
            {"name": "react", "fn": self._react_strategy},
            {"name": "direct", "fn": self._direct_strategy},
        ])
        self.degradation = GracefulDegradation()
        self.safe_executor = SafeToolExecutor(timeout_seconds=10, max_retries=1)

    def _react_strategy(self, question: str) -> Tuple[Dict, bool]:
        """Primary strategy: full ReAct loop."""
        tools_desc = "\n".join(f"- {n}: {t['description']}" for n, t in self.tools.items())
        messages = [
            {"role": "system", "content": f"You have tools:\n{tools_desc}\nUse Thought/Action/Action Input or Thought/Final Answer format. Be concise."},
            {"role": "user", "content": question},
        ]
        tools_used = []

        for step in range(self.max_steps):
            response = generate(messages, max_new_tokens=256, temperature=0.3, do_sample=True)

            final_match = re.search(r'Final Answer:\s*(.+)', response, re.IGNORECASE)
            if final_match:
                return {"answer": final_match.group(1).strip(),
                        "tools_used": tools_used, "steps": step + 1,
                        "strategy": "react"}, True

            action_match = re.search(r'Action:\s*(\w+)', response)
            input_match = re.search(r'Action Input:\s*(.+)', response)

            if action_match and input_match:
                tool_name = action_match.group(1).strip()
                tool_input = input_match.group(1).strip()
                if tool_name in self.tools:
                    result, ok = self.safe_executor.execute(
                        tool_name, self.tools[tool_name]["fn"], tool_input
                    )
                    tools_used.append(tool_name)
                    if ok:
                        self.degradation.add_result(f"tool:{tool_name}", result)
                        messages.append({"role": "assistant", "content": response})
                        messages.append({"role": "user", "content": f"Observation: {result}"})
                    else:
                        self.degradation.add_failure(f"tool:{tool_name}", str(result))
                        messages.append({"role": "assistant", "content": response})
                        messages.append({"role": "user", "content":
                            f"Observation: Tool error: {result}. Try a different approach."})
                else:
                    messages.append({"role": "assistant", "content": response})
                    messages.append({"role": "user", "content":
                        f"Observation: Tool '{tool_name}' not found. Available: {list(self.tools.keys())}"})
            else:
                # Can't parse — treat as answer
                return {"answer": response.strip().split("\n")[0],
                        "tools_used": tools_used, "steps": step + 1,
                        "strategy": "react"}, True

        return {"answer": "Max steps reached.", "tools_used": tools_used}, False

    def _direct_strategy(self, question: str) -> Tuple[Dict, bool]:
        """Fallback strategy: direct LLM answer without tools."""
        messages = [{"role": "user", "content": f"Answer concisely: {question}"}]
        response = generate(messages, max_new_tokens=128, temperature=0.3, do_sample=True)
        return {"answer": response.strip(), "tools_used": [],
                "steps": 1, "strategy": "direct"}, True

    def run(self, question: str) -> Dict:
        """Run the resilient agent on a question."""
        self.degradation = GracefulDegradation()  # reset per run
        result, strategy_name = self.fallback.execute(question)
        if result:
            return result
        # All strategies failed — return degraded result
        return {
            "answer": self.degradation.get_result().get("message", "Failed completely."),
            "tools_used": [], "steps": 0, "strategy": "degraded",
        }

print("✓ ResilientAgent defined")

In [ ]:
# Test on 5 scenarios including deliberately tricky ones
test_scenarios = [
    "What is 347 * 23?",
    "What is the capital of France?",
    "Calculate the square root of 1764.",
    "If a rectangle has area 120 and width 8, what is its length?",
    "What is the meaning of the universe, life, and everything?",
]

agent = ResilientAgent(TOOLS, max_steps=5, max_retries=2)

print("Resilient Agent Test:")
print("=" * 60)
for q in test_scenarios:
    result = agent.run(q)
    strategy = result.get("strategy", "unknown")
    tools = result.get("tools_used", [])
    steps = result.get("steps", 0)
    print(f"  Q: {q}")
    print(f"  A: {result['answer'][:80]}")
    print(f"  Strategy: {strategy} | Tools: {tools} | Steps: {steps}")
    print()

## 9. Recovery Rate Analysis

Compare agent performance with and without resilience on tasks where failures are injected.

In [ ]:
# Create a "fragile" agent without resilience for comparison
def fragile_agent(question: str) -> Dict:
    """Agent without any resilience — crashes on first error."""
    tools_desc = "\n".join(f"- {n}: {t['description']}" for n, t in TOOLS.items())
    messages = [
        {"role": "system", "content": f"Tools: {tools_desc}\nUse Thought/Action/Action Input or Final Answer."},
        {"role": "user", "content": question},
    ]
    for step in range(5):
        response = generate(messages, max_new_tokens=256, temperature=0.3, do_sample=True)
        final_match = re.search(r'Final Answer:\s*(.+)', response, re.IGNORECASE)
        if final_match:
            return {"answer": final_match.group(1).strip(), "steps": step + 1, "status": "success"}
        action_match = re.search(r'Action:\s*(\w+)', response)
        input_match = re.search(r'Action Input:\s*(.+)', response)
        if action_match and input_match:
            tool_name = action_match.group(1).strip()
            tool_input = input_match.group(1).strip()
            if tool_name in TOOLS:
                try:
                    result = TOOLS[tool_name]["fn"](tool_input)
                    messages.append({"role": "assistant", "content": response})
                    messages.append({"role": "user", "content": f"Observation: {result}"})
                except Exception as e:
                    return {"answer": f"CRASHED: {e}", "steps": step + 1, "status": "error"}
            else:
                return {"answer": f"CRASHED: unknown tool {tool_name}", "steps": step + 1, "status": "error"}
        else:
            return {"answer": response.strip().split("\n")[0], "steps": step + 1, "status": "success"}
    return {"answer": "Max steps reached", "steps": 5, "status": "timeout"}

# Test both agents on the same 10 scenarios
test_tasks = [
    "What is 347 * 23?",
    "What is the capital of France?",
    "Calculate 15% of 84.50.",
    "Who wrote 1984?",
    "What is the square root of 1764?",
    "What is the chemical symbol for gold?",
    "How many planets are in our solar system?",
    "What is (120 / 8)?",
    "What is the boiling point of water in Fahrenheit?",
    "If it takes 5 machines 5 minutes to make 5 widgets, how long for 100 machines to make 100?",
]

resilient = ResilientAgent(TOOLS)

print("Recovery Rate Comparison (10 tasks):")
print("=" * 65)
print(f"  {'Task':<55} {'Fragile':>7} {'Resilient':>9}")
print(f"  {'-'*55} {'-'*7} {'-'*9}")

fragile_successes = 0
resilient_successes = 0

for q in test_tasks:
    # Fragile
    f_result = fragile_agent(q)
    f_ok = f_result.get("status") == "success"
    if f_ok:
        fragile_successes += 1

    # Resilient
    r_result = resilient.run(q)
    r_ok = r_result.get("strategy") != "degraded"
    if r_ok:
        resilient_successes += 1

    f_icon = "✓" if f_ok else "✗"
    r_icon = "✓" if r_ok else "✗"
    print(f"  {q[:55]:<55} {f_icon:>7} {r_icon:>9}")

print(f"\n  {'TOTAL':.<55} {fragile_successes:>5}/10 {resilient_successes:>7}/10")
print(f"  Success rate: fragile={fragile_successes/10:.0%}, resilient={resilient_successes/10:.0%}")

## 10. Key Takeaways

### Resilience Patterns Summary

| Pattern | What It Does | When to Use |
|---------|-------------|-------------|
| **RetryWithFeedback** | Re-prompt with error details for self-correction | Malformed LLM output |
| **FallbackChain** | Try alternative strategies in order | Multiple valid approaches |
| **CircuitBreaker** | Stop trying after N consecutive failures | Flaky external services |
| **GracefulDegradation** | Return partial results instead of nothing | Multi-step tasks |
| **Timeout** | Kill hanging operations after time limit | External API calls |
| **SafeToolExecutor** | Combines timeout + retry + circuit breaker | All tool calls |

### Design Principles

1. **Expect failure** — design for it from the start, not as an afterthought
2. **Fail fast** — detect failures early and stop wasting resources
3. **Feedback loops** — tell the LLM what went wrong so it can self-correct
4. **Partial > nothing** — some answer is better than a crash
5. **Layer defenses** — combine multiple patterns for defense in depth
6. **Observe everything** — log every retry, fallback, and circuit state for debugging

### What's Next

In the next notebook, we build the **Model Context Protocol (MCP)** from scratch — the standard for agent-tool communication.